##Machine Learning versus Structural Gravity
##M598 - Master's Dissertation
##Thomas Luiz Reis - GH1043614

##Step 1 - Problem Statement

This notebook is the modelling core of my dissertation *"Machine Learning versus Structural Gravity: Predicting and Explaining EU Import Flows of Textile Trimmings, with an Application to Brazilian Suppliers (2015-2025)"* (Gisma University of Applied Sciences, 2026).

It answers the three research questions of the thesis on a single panel:

- RQ1: do machine learning models predict EU imports of textile trimmings at product level better than a structural gravity model? After the discussion with my supervisor the design also gained a hybrid entry, a model that feeds the gravity model's own predictions into the machine learning side.
- RQ2: do the gravity coefficients and the machine learning importances agree on what drives these flows?
- RQ3: does Brazil trade below the level the models expect, compared with suppliers of similar size?

Flows at this resolution are sparse (about 74% of the panel rows are zeros) and very persistent, which makes the contest between economic structure and algorithmic flexibility less obvious than it sounds.

###Data collection
The panel merges four public sources: Eurostat Comext for the EU import values, ComexStat/MDIC for the Brazilian mirror records, World Bank WDI for GDP and population, and the CEPII Gravity database for distance and the other pair covariates. The merged file loads straight from my repo, where the full pipeline lives:
https://github.com/THS-99/trimmings-gravity-vs-ml

Two honest notes. This is a lighter version of scripts 07 to 13 of the repo, so the numbers here can differ a little from the exact ones in the thesis tables. And two pieces stay out because they need hours of compute: the SHAP directions and the nine robustness checks, both in the repo. The rest of the notebook runs in around 15 minutes on a normal Colab CPU runtime.

##Step 2 - Import Libraries

In [ ]:
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42

##Step 3 - Data Exploration

One row is one combination of exporter, EU destination, HS6 product and year, 2015 to 2025, with zeros kept as real zeros (no trade that year).

In [ ]:
url = "https://raw.githubusercontent.com/THS-99/trimmings-gravity-vs-ml/main/data/processed/panel_trimmings_2015_2025.csv.gz"
df = pd.read_csv(url, dtype={"hs6": str, "heading": str})
print(df.shape)
df.head()

In [ ]:
# quick sanity checks
print("years:", df.year.min(), "to", df.year.max())
print("exporters:", df.exporter.nunique(), "| destinations:", df.destination.nunique(), "| products:", df.hs6.nunique())
print("share of zero flows:", round((df.value_eur == 0).mean(), 3))

In [ ]:
imports_year = df.groupby("year").value_eur.sum() / 1e6
imports_year.plot(marker="o", figsize=(7, 4))
plt.ylabel("million EUR")
plt.title("EU imports of the six trimmings headings, panel total")
plt.tight_layout()
plt.show()

The 2020 dip and the 2021 rebound show up exactly where they should, which was one of the real-world validation checks my supervisor asked for. The test years (2023 to 2025) sit above every pre-pandemic level, so the models are evaluated on a market at historically high levels and not on a quiet tail.

##Step 4 - Data Preprocessing & Feature Engineering

Trade flows are very persistent: a flow that was big last year is probably big this year too. So besides the gravity variables (GDP, population, distance, common language and so on) I give the models the recent history of each flow: the last two values in logs, a 3-year rolling mean, and how many years in a row the flow has been zero. Everything is computed strictly from past years, so nothing leaks from the future.

The target is log1p of the import value, because the values are extremely skewed and there are lots of zeros. Exporter, destination and product identities go in as one-hot dummies, which is the machine learning version of the fixed effects the gravity model uses.

In [ ]:
df = df.sort_values(["exporter", "destination", "hs6", "year"]).reset_index(drop=True)
df["log_value"] = np.log1p(df["value_eur"])

# Taiwan has no GDP in the WDI after 2021, I carry the last value forward
df["gdp_exporter"] = df.groupby("exporter")["gdp_exporter"].ffill()
df["pop_exporter"] = df.groupby("exporter")["pop_exporter"].ffill()

grp = df.groupby(["exporter", "destination", "hs6"])["log_value"]
df["lag1"] = grp.shift(1)
df["lag2"] = grp.shift(2)
df["roll3"] = grp.transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())

# how many years in a row the flow was zero before year t
def streak_before(values):
    out, run = [], 0
    for v in values:
        out.append(run)
        run = run + 1 if v == 0 else 0
    return out

df["zero_streak"] = (df.groupby(["exporter", "destination", "hs6"])["value_eur"]
                        .transform(lambda s: pd.Series(streak_before(s.values), index=s.index)))

hist_cols = ["lag1", "lag2", "roll3", "zero_streak"]
df[hist_cols] = df[hist_cols].fillna(0)

In [ ]:
# log the skewed gravity variables, keep the dummies as they are
for c in ["dist", "gdp_exporter", "pop_exporter", "gdp_destination", "pop_destination"]:
    df["log_" + c] = np.log(df[c])

gravity_cols = ["log_dist", "contig", "comlang_off", "comcol", "rta",
                "log_gdp_exporter", "log_pop_exporter", "log_gdp_destination", "log_pop_destination"]

X_full = pd.get_dummies(df[gravity_cols + hist_cols + ["exporter", "destination", "hs6"]],
                        columns=["exporter", "destination", "hs6"], dtype=float)
X_full = X_full.astype("float32")  # the tree models use float32 anyway, and this halves the memory
y = df["log_value"]
years = df["year"]
print(X_full.shape)

##Step 5 - Model Training

Six things get compared, from dumbest to fanciest:

1. a naive baseline that repeats last year's value (surprisingly hard to beat in trade data),
2. the structural gravity model, estimated with PPML in its prediction form (exporter, destination and product fixed effects plus the time-varying covariates). This is the baseline of the whole thesis,
3. a random forest,
4. LightGBM,
5. a hybrid: the same random forest, with the gravity model's prediction added as one extra feature. This was my supervisor's suggestion, and it asks a precise question: once the model knows the history of a flow, does economic structure still add anything?
6. the random forest again but *without* the history features, to see how much of the performance actually comes from them.

There is also a second PPML specification, the interpretable one, which I only use for RQ2. It keeps the gravity covariates explicit and absorbs time and product effects, so its coefficients read as elasticities. Standard errors are clustered by exporter-destination pair, because the same corridor shows up eleven times in the panel and its errors are not independent.

The hyperparameters are the winners of a grid search that lives in the repo (script 09 and `results/ml_tuning_report.json`), I just reuse them here.

In [ ]:
def rmse_mae(y_true_eur, y_pred_eur):
    rmse = np.sqrt(mean_squared_error(y_true_eur, y_pred_eur))
    mae = mean_absolute_error(y_true_eur, y_pred_eur)
    return rmse, mae

def from_log(pred_log):
    return np.clip(np.expm1(pred_log), 0, None)  # back to euros, no negative trade

def fit_ppml(train_df, test_df):
    cols = gravity_cols + ["exporter", "destination", "hs6"]
    Xtr = pd.get_dummies(train_df[cols], columns=["exporter", "destination", "hs6"],
                         drop_first=True, dtype=float)
    Xte = pd.get_dummies(test_df[cols], columns=["exporter", "destination", "hs6"],
                         drop_first=True, dtype=float)
    Xte = Xte.reindex(columns=Xtr.columns, fill_value=0.0)
    Xtr = sm.add_constant(Xtr, has_constant="add")
    Xte = sm.add_constant(Xte, has_constant="add")
    res = sm.GLM(train_df.value_eur.values, Xtr.values, family=sm.families.Poisson()).fit(maxiter=300)
    print("  PPML converged:", res.converged)
    return res.predict(Xte.values)

##Step 6 - Experiments

The models never see the test year in training. I use three rolling origins: train up to 2022 and test on 2023, train up to 2023 and test on 2024, train up to 2024 and test on 2025. Errors are measured in euros, because that is the unit that actually matters.

The PPML fit does double duty: it is scored as a model on its own, and its predictions become the extra feature of the hybrid. For the training rows I do not use the in-sample fitted values, because those were fitted on the very targets the forest then learns from, which is a form of leakage my supervisor pointed out. Instead the feature for year t always comes from a gravity model fitted on the years before t, for training and test rows alike, so both carry the same kind of one-year-ahead prediction. The first two years of the panel have no fit before them and get a zero, like the second lag.

Besides the error metrics I keep the row-level predictions of the three models that matter later (gravity, forest, hybrid). Steps 7 and 9 need them for the significance test and for the Brazil analysis.

In [ ]:
test_years = [2023, 2024, 2025]
results = []
kept_preds = []
X_nohist = X_full.drop(columns=hist_cols)

rf_params = dict(n_estimators=200, min_samples_leaf=5, max_features=0.5, max_samples=0.7,
                 random_state=SEED, n_jobs=-1)

# gravity prediction as a feature: for every year t, a PPML fit on the years before t
# (starting with two years of history, so 2015 and 2016 get 0 like the second lag)
ppml_feat = pd.Series(0.0, index=df.index)
for t in range(2017, 2026):
    ppml_feat[years == t] = np.log1p(fit_ppml(df[years < t], df[years == t]))

for test_year in test_years:
    train = years < test_year
    test = years == test_year
    y_eur = df.loc[test, "value_eur"].values

    # 1. naive baseline: last year's value
    results.append(["naive (last year)", test_year, *rmse_mae(y_eur, from_log(df.loc[test, "lag1"]))])

    # 2. PPML gravity
    ppml_te = fit_ppml(df[train], df[test])
    results.append(["PPML gravity", test_year, *rmse_mae(y_eur, ppml_te)])

    # 3. random forest
    rf = RandomForestRegressor(**rf_params)
    rf.fit(X_full[train], y[train])
    pred_rf = from_log(rf.predict(X_full[test]))
    results.append(["random forest", test_year, *rmse_mae(y_eur, pred_rf)])

    # 4. LightGBM
    lgbm = LGBMRegressor(n_estimators=1500, learning_rate=0.05, num_leaves=63,
                         random_state=SEED, n_jobs=-1, verbose=-1)
    lgbm.fit(X_full[train], y[train])
    results.append(["lightgbm", test_year, *rmse_mae(y_eur, from_log(lgbm.predict(X_full[test])))])

    # 5. hybrid: random forest + the gravity prediction as a feature
    X_hyb = X_full.copy()
    X_hyb["ppml_pred_log"] = ppml_feat.astype("float32")
    rf_hyb = RandomForestRegressor(**rf_params)
    rf_hyb.fit(X_hyb[train], y[train])
    pred_hyb = from_log(rf_hyb.predict(X_hyb[test]))
    results.append(["hybrid (RF + gravity)", test_year, *rmse_mae(y_eur, pred_hyb)])

    # 6. random forest without the history features
    rf_nohist = RandomForestRegressor(**rf_params)
    rf_nohist.fit(X_nohist[train], y[train])
    results.append(["random forest, no history", test_year, *rmse_mae(y_eur, from_log(rf_nohist.predict(X_nohist[test])))])

    kept_preds.append(pd.DataFrame({
        "exporter": df.loc[test, "exporter"].values,
        "destination": df.loc[test, "destination"].values,
        "heading": df.loc[test, "heading"].values,
        "year": test_year,
        "observed": y_eur,
        "pred_ppml": ppml_te,
        "pred_rf": pred_rf,
        "pred_hybrid": pred_hyb}))

    print("done with", test_year)

preds = pd.concat(kept_preds, ignore_index=True)

# from here on only the hybrid forest is needed, and these objects are big
del rf, rf_nohist, lgbm, X_nohist
gc.collect()

In [ ]:
res = pd.DataFrame(results, columns=["model", "test_year", "rmse_eur", "mae_eur"])
res.pivot(index="model", columns="test_year", values="rmse_eur").round(0)

In [ ]:
# average over the three test years
res.groupby("model")[["rmse_eur", "mae_eur"]].mean().round(0).sort_values("rmse_eur")

##Step 7 - Model Assessment (RQ1)

Averages are a start, but they do not say whether a gap is real. Two of the differences in the table deserve a test, so I run a paired bootstrap, both models scored on exactly the same rows, and look at the distribution of the RMSE difference. The resampling unit is the exporter-destination pair and not the single row, because the rows of one corridor (its products and years) are correlated with each other and treating them as independent would make the intervals too narrow. The first comparison is the headline of the thesis, machine learning against gravity. The second is my supervisor's question, whether the hybrid actually beats the plain forest.

In [ ]:
def paired_bootstrap(observed, pred_a, pred_b, clusters, n_boot=2000):
    # positive difference means model A has the lower error; clusters are resampled, not rows
    err = pd.DataFrame({"a": (pred_a - observed) ** 2, "b": (pred_b - observed) ** 2, "cl": clusters})
    g = err.groupby("cl")
    sum_a, sum_b, n = g.a.sum().values, g.b.sum().values, g.size().values
    rng = np.random.default_rng(SEED)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, len(n), len(n))
        diffs[i] = np.sqrt(sum_b[idx].sum() / n[idx].sum()) - np.sqrt(sum_a[idx].sum() / n[idx].sum())
    observed_diff = np.sqrt(err.b.mean()) - np.sqrt(err.a.mean())
    p_value = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return observed_diff, np.percentile(diffs, [2.5, 97.5]), max(p_value, 1 / n_boot)

obs = preds.observed.values
pairs = (preds.exporter + "_" + preds.destination).values

for label, a, b in [("hybrid vs PPML gravity", preds.pred_hybrid.values, preds.pred_ppml.values),
                    ("hybrid vs plain random forest", preds.pred_hybrid.values, preds.pred_rf.values)]:
    diff, ci, p = paired_bootstrap(obs, a, b, pairs)
    print(f"{label}: RMSE difference {diff:,.0f} EUR, 95% CI [{ci[0]:,.0f}; {ci[1]:,.0f}], p = {p:.3f}")

So the two answers are different in kind. Against gravity the gap is large and clearly significant, the hybrid forest cuts the error by more than half. Against the plain forest the gap is a few hundred euros with a confidence interval that crosses zero, which is a tie. Adding the gravity prediction to a model that already sees the history of each flow changes nothing that survives resampling.

Where the gravity feature does pay is when the history is gone: in the repo run, the no-history forest improves from about EUR 340k to 313k of RMSE once the PPML prediction is added, and the naive baseline in the table above shows why. These flows repeat themselves, and the lags already carry most of what the structure would tell you.

In [ ]:
imp = pd.Series(rf_hyb.feature_importances_, index=X_hyb.columns).sort_values().tail(15)
imp.plot.barh(figsize=(7, 5))
plt.title("Hybrid random forest feature importance (top 15)")
plt.tight_layout()
plt.show()

The picture is one-sided. The 3-year rolling mean and the first lag dominate, and the gravity prediction (ppml_pred_log) shows up next, ahead of GDP and every other economic variable. The forest reads the gravity model's opinion, ranks it above the raw covariates, and still takes almost all of its accuracy from the recent history of each flow.

##Step 8 - Explanatory Analysis (RQ2)

Prediction is not understanding, so this step puts the two paradigms side by side on the same variables. On one side the gravity elasticities from the interpretable PPML specification. On the other the permutation importance of the winning forest, measured on held-out data: how much accuracy the model loses when that one column is shuffled. An importance is not a coefficient, it says the model uses a variable, not which way the outcome moves, and that difference is the point of the comparison.

Both are computed on the longest origin (trained through 2024, tested on 2025). The permutation part runs on a sample of the test year to keep it inside a few minutes.

In [ ]:
specA_cols = ["log_gdp_exporter", "log_gdp_destination", "log_dist",
              "contig", "comlang_off", "comcol", "rta"]

train_o3 = years < 2025
test_o3 = years == 2025

X_a = pd.get_dummies(df.loc[train_o3, specA_cols + ["year", "heading"]],
                     columns=["year", "heading"], drop_first=True, dtype=float)
X_a = sm.add_constant(X_a, has_constant="add")
pairs = df.loc[train_o3, "exporter"] + "_" + df.loc[train_o3, "destination"]

ppml_a = sm.GLM(df.loc[train_o3, "value_eur"].values, X_a.values,
                family=sm.families.Poisson()).fit(maxiter=300, cov_type="cluster",
                                                  cov_kwds={"groups": pairs.values})
coefs = pd.DataFrame({"variable": X_a.columns, "gravity_coef": ppml_a.params,
                      "p_value": ppml_a.pvalues})
coefs = coefs[coefs.variable.isin(specA_cols)]

In [ ]:
samp = X_hyb[test_o3].sample(3000, random_state=SEED)
# one worker on purpose: with n_jobs=-1 joblib copies the whole forest per worker
perm = permutation_importance(rf_hyb, samp, y[samp.index], n_repeats=2,
                              random_state=SEED, n_jobs=1)
perm_imp = pd.Series(perm.importances_mean, index=X_hyb.columns)

coefs["ml_perm_importance"] = coefs.variable.map(perm_imp)
coefs.round(3)

In [ ]:
# the same importances, now for the features the gravity model cannot see
perm_imp[hist_cols + ["ppml_pred_log"]].sort_values(ascending=False).round(3)

The two views agree on one thing and disagree on the rest. Both say market size matters: the GDP elasticities are close to one and significant, and exporter GDP is the strongest economic variable in the forest too. Distance is weak on both sides, which is already unusual for a gravity model and fits what the textile literature reports.

Two coefficients come out with a sign nobody would predict. Common colonizer is negative, and so is the trade agreement dummy. I read the second one as composition rather than causation: the EU's largest trimmings suppliers (China, Taiwan, Japan, the United States) sell without a preferential agreement, while many agreement partners are small suppliers, and without pair fixed effects the dummy absorbs that selection.

The real divergence is in the second table. Every economic variable is an order of magnitude behind the rolling mean and the first lag. The gravity structure has no way of seeing that a corridor was already running last year, and in this niche that is most of the story. Incumbency, not distance, is the barrier.

##Step 9 - Brazil (RQ3)

The last question is applied. If the models can predict what a supplier of Brazil's characteristics should be selling to the EU, then the gap between that prediction and reality is a signal worth reading. A single deviation number would not mean much, since it mixes unrealized potential with model error, so the comparison is against the ten suppliers closest to Brazil in observed size. Whatever error the model makes on small suppliers, it makes for all of them, and what is left is the ranking.

In [ ]:
by_exporter = preds.groupby("exporter")[["observed", "pred_hybrid"]].sum()
by_exporter["deviation_pct"] = 100 * (by_exporter.observed - by_exporter.pred_hybrid) / by_exporter.pred_hybrid

# the ten exporters closest to Brazil in observed test-period size
brazil_size = by_exporter.loc["BR", "observed"]
peers = (by_exporter.observed - brazil_size).abs().drop("BR").sort_values().head(10).index

peer_table = by_exporter.loc[list(peers) + ["BR"]].sort_values("deviation_pct")
(peer_table / [1e6, 1e6, 1]).round(2)

In [ ]:
print("Brazil deviation under the hybrid forest:", round(by_exporter.loc["BR", "deviation_pct"], 1), "%")
print("median of the ten comparable suppliers:", round(peer_table.drop("BR").deviation_pct.median(), 1), "%")

# same measure under the structural benchmark, as a cross-check
br = preds[preds.exporter == "BR"]
print("Brazil deviation under PPML gravity:",
      round(100 * (br.observed.sum() - br.pred_ppml.sum()) / br.pred_ppml.sum(), 1), "%")

In [ ]:
# where the gravity model expects Brazilian trimmings to go, and where they actually go
by_dest = br.groupby("destination")[["observed", "pred_ppml"]].sum() / 1e6
by_dest["gap_meur"] = by_dest.pred_ppml - by_dest.observed
by_dest.sort_values("gap_meur", ascending=False).head(8).round(2)

Brazil sells more than the forest expects, and less than the gravity model expects, which sounds like a contradiction until you look at the peers. Every comparable supplier beats its own prediction by more than Brazil does, so Brazil comes last in the group. That is the finding I trust, because it survives whatever bias the models share on small exporters.

The destination table is the sharpest part of the section. Almost all of Brazil's observed EU trade is a single corridor, narrow woven fabrics to Romania, while the structural model puts the unrealized volume in Portugal, Italy, France, Germany and Spain. The absence is concentrated exactly in the large Western markets where the structure says Brazil should be selling. None of this is a market entry recommendation, it is a shortlist of places worth looking at with cost and capacity data the trade statistics cannot see.

##Step 10 - Final Discussion

- Strengths: the comparison is fair by construction, every model sees the same rows and the same inputs under the same rolling-origin protocol, and the gravity baseline converged in every fit (the closest study in agriculture had to move its PPML reference to an appendix for that reason). The headline gap is large, tested, and stable across the three test years.

- Limitations: the design is one-step-ahead, so nothing here says what happens at longer horizons, where gravity models tend to hold their ground. Errors are measured in euros, so the biggest corridors dominate the metric. The hybrid was tested in its simplest form, one extra feature. And the RQ3 deviations mix potential with model error, which is why the ranking rather than the raw percentage is what I report.

- Business implications: for an EU buyer or a supplier planning where to sell, next year's trade looks a lot like this year's. Incumbency is the strongest force in this niche, and it cuts both ways: forecasts are reliable, doors open slowly. The useful output is not the forecast itself but the gap between observed and predicted flows.

- Future work: multi-step horizons, tariffs as an explicit covariate, and richer hybrids (residual learning instead of one stacked feature). All three come back in section 6.3 of the thesis.

What is *not* in this notebook: the SHAP directions, which give the sign of each contribution, and the nine robustness checks. Both need hours of compute and live in the repo: https://github.com/THS-99/trimmings-gravity-vs-ml

## References

Breiman, L. (2001) 'Random forests', *Machine Learning*, 45(1), pp. 5-32.

Conte, M., Cotterlaz, P. and Mayer, T. (2022) 'The CEPII gravity database', CEPII Working Paper No. 2022-05.

Eurostat (2026) *Comext: EU trade since 1988 by HS2-4-6 and CN8 (DS-045409)*. Available at: https://ec.europa.eu/eurostat/comext/ (Accessed: 21 August 2026).

Gopinath, M., Batarseh, F.A., Beckman, J., Kulkarni, A. and Jeong, S. (2021) 'International agricultural trade forecasting using machine learning', *Data & Policy*, 3, e1.

Harris, C.R. et al. (2020) 'Array programming with NumPy', *Nature*, 585(7825), pp. 357-362.

Hunter, J.D. (2007) 'Matplotlib: a 2D graphics environment', *Computing in Science & Engineering*, 9(3), pp. 90-95.

Ke, G. et al. (2017) 'LightGBM: a highly efficient gradient boosting decision tree', *Advances in Neural Information Processing Systems*, 30, pp. 3146-3154.

McKinney, W. (2010) 'Data structures for statistical computing in Python', *Proceedings of the 9th Python in Science Conference*, pp. 56-61.

Morland, C., Tandetzki, J. and Schier, F. (2025) 'An evaluation of gravity models and artificial neuronal networks on bilateral trade flows in wood markets', *Forest Policy and Economics*, 172, 103457.

Pedregosa, F. et al. (2011) 'Scikit-learn: machine learning in Python', *Journal of Machine Learning Research*, 12, pp. 2825-2830.

Santos Silva, J.M.C. and Tenreyro, S. (2006) 'The log of gravity', *The Review of Economics and Statistics*, 88(4), pp. 641-658.

Seabold, S. and Perktold, J. (2010) 'statsmodels: econometric and statistical modeling with Python', *Proceedings of the 9th Python in Science Conference*, pp. 92-96.

Yotov, Y.V., Piermartini, R., Monteiro, J.-A. and Larch, M. (2016) *An Advanced Guide to Trade Policy Analysis: The Structural Gravity Model*. Geneva: WTO/UNCTAD.